# Task 2 — Data Discovery, Profiling and Cleaning- Hanin

This notebook loads the raw economic indicator datasets, profiles their structure and quality, identifies data issues, cleans the data, and saves the cleaned output to `data/interim/cleaned.csv`.

In [1]:
# Import pandas for loading, profiling, and cleaning the datasets

import pandas as pd
from pathlib import Path

In [2]:
# Define the folders used in this task

RAW_DATA_PATH = Path("../data/raw")
INTERIM_DATA_PATH = Path("../data/interim")

# Create the interim folder if it does not already exist
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
# Define the path for each raw dataset

gdp_file = RAW_DATA_PATH / "Gross domestic product (GDP) at current price by Main ISIC_data.csv"

inflation_file = RAW_DATA_PATH / "Annual Change in Consumer Prices (National, %)_data.csv"

unemployment_file = RAW_DATA_PATH / "Unemployment_rate_by_nationality_and_sex_2013_2026.csv"

In [4]:
# Load the raw GDP CSV file
gdp_raw = pd.read_csv(gdp_file)

print("GDP shape:", gdp_raw.shape)

gdp_raw.head()

GDP shape: (390, 4)


,Item Code,Main Activities,Year / Quarter,Gross Domestic Product
0,1,Oil Activities,2011-Q3,"327,680"
1,1,Oil Activities,2018-Q2,"276,085"
2,1,Oil Activities,2021-Q3,"282,999"
3,1,Oil Activities,2011-Q1,"283,150"
4,1,Oil Activities,2018-Q3,"286,401"


## GDP Structure Check--Hanin

The raw GDP source is a CSV file with structured columns.

The data is already in tabular format, so no JSON flattening is required before profiling and cleaning.

In [5]:
# Check the GDP dataset structure

print("GDP shape:", gdp_raw.shape)
print("GDP columns:", gdp_raw.columns.tolist())

gdp_raw.head()

GDP shape: (390, 4)
GDP columns: ['Item Code', 'Main Activities', 'Year / Quarter', 'Gross Domestic Product']


,Item Code,Main Activities,Year / Quarter,Gross Domestic Product
0,1,Oil Activities,2011-Q3,"327,680"
1,1,Oil Activities,2018-Q2,"276,085"
2,1,Oil Activities,2021-Q3,"282,999"
3,1,Oil Activities,2011-Q1,"283,150"
4,1,Oil Activities,2018-Q3,"286,401"


## GDP Data Profiling - Hanin

The raw GDP dataset is profiled to understand its structure and identify possible data quality issues before cleaning.

In [6]:
## Display the number of rows and columns

print("Number of rows:", gdp_raw.shape[0])
print("Number of columns:", gdp_raw.shape[1])

Number of rows: 390
Number of columns: 4


In [7]:
# Display column names and data types

gdp_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Item Code               390 non-null    int64
 1   Main Activities         390 non-null    str  
 2   Year / Quarter          390 non-null    str  
 3   Gross Domestic Product  390 non-null    str  
dtypes: int64(1), str(3)
memory usage: 12.3 KB


In [8]:
# Check the number of missing values in each column
gdp_raw.isnull().sum()

Item Code                 0
Main Activities           0
Year / Quarter            0
Gross Domestic Product    0
dtype: int64

In [9]:
# Check the number of unique values in each column
gdp_raw.nunique()

Item Code                   6
Main Activities             6
Year / Quarter             65
Gross Domestic Product    389
dtype: int64

In [10]:
# Check for duplicate rows
print("Duplicate rows:", gdp_raw.duplicated().sum())

Duplicate rows: 0


In [11]:
# Create a profiling summary for each GDP column

gdp_profile = pd.DataFrame({
    "column_name": gdp_raw.columns,
    "dtype": [gdp_raw[col].dtype for col in gdp_raw.columns],
    "null_count": [gdp_raw[col].isnull().sum() for col in gdp_raw.columns],
    "null_percentage": [(gdp_raw[col].isnull().mean() * 100) for col in gdp_raw.columns],
    "unique_count": [gdp_raw[col].nunique() for col in gdp_raw.columns],
    "sample_value": [gdp_raw[col].iloc[0] for col in gdp_raw.columns]
})

gdp_profile

,column_name,dtype,null_count,null_percentage,unique_count,sample_value
0,Item Code,int64,0,0.0,6,1
1,Main Activities,str,0,0.0,6,Oil Activities
2,Year / Quarter,str,0,0.0,65,2011-Q3
3,Gross Domestic Product,str,0,0.0,389,"327,680"


### GDP Issues and Cleaning Decisions

| Issue                                               | Decision                                                   |
| --------------------------------------------------- | ---------------------------------------------------------- |
| Source data is already in CSV tabular format        | No flattening is required                                  |
| Missing values                                      | No action required if no missing values are found          |
| Duplicate rows                                      | No action required if no duplicate rows are found          |
| Column names contain spaces and special characters  | Convert column names to lowercase snake_case               |
| Dataset contains multiple Main ISIC activities      | Keep only the `Gross Domestic Product` record              |
| Dataset contains data before 2013                   | Keep records from 2013 onward                              |
| `Year / Quarter` contains year and quarter together | Split it into `year`, `quarter`, and create `year_quarter` |


In [12]:
# Create a copy of the raw GDP data before cleaning
gdp_clean = gdp_raw.copy()

# Rename columns to clear and simple names
gdp_clean = gdp_clean.rename(
    columns={
        "Item Code": "item_code",
        "Main Activities": "main_activities",
        "Year / Quarter": "year_quarter",
        "Gross Domestic Product": "gdp_value"
    }
)

# Keep only total GDP records
gdp_clean = gdp_clean[
    gdp_clean["main_activities"] == "Gross Domestic Product"
].copy()

# Split year_quarter into year and quarter
gdp_clean[["year", "quarter"]] = (
    gdp_clean["year_quarter"]
    .str.split("-", expand=True)
)

# Convert year to integer
gdp_clean["year"] = gdp_clean["year"].astype(int)

# Keep data from 2013 onward
gdp_clean = gdp_clean[
    gdp_clean["year"] >= 2013
].copy()

# Convert GDP values from text with commas to numeric
gdp_clean["gdp_value"] = (
    gdp_clean["gdp_value"]
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Keep only the columns needed for analysis
gdp_clean = gdp_clean[
    [
        "main_activities",
        "year",
        "quarter",
        "year_quarter",
        "gdp_value"
    ]
]

# Sort the data chronologically
gdp_clean = gdp_clean.sort_values(
    by=["year", "quarter"]
).reset_index(drop=True)

# Display the cleaned GDP data
print("Cleaned GDP shape:", gdp_clean.shape)

gdp_clean.head()

Cleaned GDP shape:

 (53, 5)


,main_activities,year,quarter,year_quarter,gdp_value
0,Gross Domestic Product,2013,Q1,2013-Q1,712484.0
1,Gross Domestic Product,2013,Q2,2013-Q2,721647.0
2,Gross Domestic Product,2013,Q3,2013-Q3,729632.0
3,Gross Domestic Product,2013,Q4,2013-Q4,722821.0
4,Gross Domestic Product,2014,Q1,2014-Q1,762832.0


## Inflation Data - Hanin

The raw inflation dataset is loaded and profiled before applying any cleaning steps.

In [13]:

# Load the inflation CSV and skip the metadata rows
inflation_raw = pd.read_csv(
    inflation_file,
    skiprows=4
)

# Display the dataset shape and first rows
print("Inflation shape:", inflation_raw.shape)

inflation_raw.head()

Inflation shape: (89536, 7)


,Item Code,Item Name,YEAR_MONTH,CL_ADMIN_PRICES_E,D_BASKET_LVL,Indicator Name,OBS_VALUE
0,01.1.6.5.2,Fresh kiwi,9/1/17,Index Numbers,5,Selected period with the corresponding period ...,0.0
1,01.1.6.5.2,Fresh kiwi,5/1/25,Index Numbers,5,Selected period with the corresponding period ...,9.9
2,01.1.6.5.2,Fresh kiwi,4/1/25,Index Numbers,5,Selected period with the corresponding period ...,7.2
3,01.1.6.5.2,Fresh kiwi,3/1/25,Index Numbers,5,Selected period with the corresponding period ...,11.8
4,01.1.6.5.2,Fresh kiwi,2/1/25,Index Numbers,5,Selected period with the corresponding period ...,16.1


In [14]:
# Check the inflation dataset structure and data types
print("Number of rows:", inflation_raw.shape[0])
print("Number of columns:", inflation_raw.shape[1])

inflation_raw.info()

Number of rows: 89536
Number of columns: 7
<class 'pandas.DataFrame'>
RangeIndex: 89536 entries, 0 to 89535
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Item Code          89536 non-null  str    
 1   Item Name          89536 non-null  str    
 2   YEAR_MONTH         89536 non-null  str    
 3   CL_ADMIN_PRICES_E  89536 non-null  str    
 4   D_BASKET_LVL       89536 non-null  int64  
 5   Indicator Name     89536 non-null  str    
 6   OBS_VALUE          89536 non-null  float64
dtypes: float64(1), int64(1), str(5)
memory usage: 4.8 MB


In [15]:
# Check missing values in each column
inflation_raw.isnull().sum()

Item Code            0
Item Name            0
YEAR_MONTH           0
CL_ADMIN_PRICES_E    0
D_BASKET_LVL         0
Indicator Name       0
OBS_VALUE            0
dtype: int64

In [16]:
# Check the number of unique values in each column
inflation_raw.nunique()

Item Code            736
Item Name            547
YEAR_MONTH           151
CL_ADMIN_PRICES_E      1
D_BASKET_LVL           5
Indicator Name         1
OBS_VALUE            771
dtype: int64

In [17]:
# Check duplicate rows
print("Duplicate rows:", inflation_raw.duplicated().sum())

Duplicate rows: 0


In [18]:
# Create a profiling summary for the inflation dataset
inflation_profile = pd.DataFrame({
    "column_name": inflation_raw.columns,
    "dtype": [inflation_raw[col].dtype for col in inflation_raw.columns],
    "null_count": [inflation_raw[col].isnull().sum() for col in inflation_raw.columns],
    "null_percentage": [inflation_raw[col].isnull().mean() * 100 for col in inflation_raw.columns],
    "unique_count": [inflation_raw[col].nunique() for col in inflation_raw.columns],
    "sample_value": [inflation_raw[col].iloc[0] for col in inflation_raw.columns]
})

inflation_profile

,column_name,dtype,null_count,null_percentage,unique_count,sample_value
0,Item Code,str,0,0.0,736,01.1.6.5.2
1,Item Name,str,0,0.0,547,Fresh kiwi
2,YEAR_MONTH,str,0,0.0,151,9/1/17
3,CL_ADMIN_PRICES_E,str,0,0.0,1,Index Numbers
4,D_BASKET_LVL,int64,0,0.0,5,5
5,Indicator Name,str,0,0.0,1,Selected period with the corresponding period ...
6,OBS_VALUE,float64,0,0.0,771,0.0


In [19]:
# Find the General Index records used for headline inflation
general_index = inflation_raw[
    inflation_raw["Item Name"].str.contains(
        "General",
        case=False,
        na=False
    )
][["Item Code", "Item Name", "D_BASKET_LVL"]].drop_duplicates()

general_index

,Item Code,Item Name,D_BASKET_LVL
159,0,General Index,1


In [20]:
# Create a copy containing only the General Index records
inflation_clean = inflation_raw[
    inflation_raw["Item Code"].astype(str) == "0"
].copy()

# Check the filtered data
print("Inflation records:", inflation_clean.shape[0])

inflation_clean.head()

Inflation records: 151


,Item Code,Item Name,YEAR_MONTH,CL_ADMIN_PRICES_E,D_BASKET_LVL,Indicator Name,OBS_VALUE
159,0,General Index,7/1/20,Index Numbers,1,Selected period with the corresponding period ...,6.1
160,0,General Index,6/1/20,Index Numbers,1,Selected period with the corresponding period ...,0.0
161,0,General Index,5/1/20,Index Numbers,1,Selected period with the corresponding period ...,0.5
162,0,General Index,4/1/20,Index Numbers,1,Selected period with the corresponding period ...,0.9
163,0,General Index,3/1/20,Index Numbers,1,Selected period with the corresponding period ...,1.1


In [21]:
# Convert YEAR_MONTH to datetime
# The source date format is Month/Day/Year, for example: 7/1/20 = July 1, 2020
inflation_clean["YEAR_MONTH"] = pd.to_datetime(
    inflation_clean["YEAR_MONTH"],
    format="%m/%d/%y"
)

# Check the date range
print("First date:", inflation_clean["YEAR_MONTH"].min())
print("Last date:", inflation_clean["YEAR_MONTH"].max())

inflation_clean[["YEAR_MONTH", "OBS_VALUE"]].head()

First date: 2014-01-01 00:00:00
Last date: 2026-07-01 00:00:00


,YEAR_MONTH,OBS_VALUE
159,2020-07-01,6.1
160,2020-06-01,0.0
161,2020-05-01,0.5
162,2020-04-01,0.9
163,2020-03-01,1.1


In [22]:
# Keep inflation data from 2013 onward
inflation_clean = inflation_clean[
    inflation_clean["YEAR_MONTH"].dt.year >= 2013
].copy()

# Sort the data chronologically
inflation_clean = inflation_clean.sort_values("YEAR_MONTH")

# Reset the index
inflation_clean = inflation_clean.reset_index(drop=True)

# Check the date range after filtering
print("First date:", inflation_clean["YEAR_MONTH"].min())
print("Last date:", inflation_clean["YEAR_MONTH"].max())
print("Number of records:", inflation_clean.shape[0])

First date: 2014-01-01 00:00:00
Last date: 2026-07-01 00:00:00
Number of records: 151


In [23]:
# Standardize column names
inflation_clean.columns = (
    inflation_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Keep the columns needed for the inflation indicator
# Keep the columns needed for analysis and future joins
inflation_clean = inflation_clean[
    [
        "item_name",
        "year_month",
        "obs_value"
    ]
].copy()

# Rename columns to clearer names
inflation_clean = inflation_clean.rename(
    columns={
        "year_month": "date",
        "obs_value": "inflation_rate"
    }
)

# Sort by date
inflation_clean = inflation_clean.sort_values("date").reset_index(drop=True)

inflation_clean.head()

,item_name,date,inflation_rate
0,General Index,2014-01-01,1.1
1,General Index,2014-02-01,1.5
2,General Index,2014-03-01,1.5
3,General Index,2014-04-01,2.0
4,General Index,2014-05-01,2.0


In [24]:
# Check duplicate rows after cleaning
print("Duplicate rows:", inflation_clean.duplicated().sum())

# Check final data types
inflation_clean.info()

Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   item_name       151 non-null    str           
 1   date            151 non-null    datetime64[us]
 2   inflation_rate  151 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 3.7 KB


In [25]:
# Add time columns for future quarterly transformation and joins
inflation_clean["year"] = inflation_clean["date"].dt.year

inflation_clean["quarter"] = (
    "Q" + inflation_clean["date"].dt.quarter.astype(str)
)

inflation_clean["year_quarter"] = (
    inflation_clean["year"].astype(str)
    + "-"
    + inflation_clean["quarter"]
)

# Display the updated cleaned inflation data
inflation_clean.head()

,item_name,date,inflation_rate,year,quarter,year_quarter
0,General Index,2014-01-01,1.1,2014,Q1,2014-Q1
1,General Index,2014-02-01,1.5,2014,Q1,2014-Q1
2,General Index,2014-03-01,1.5,2014,Q1,2014-Q1
3,General Index,2014-04-01,2.0,2014,Q2,2014-Q2
4,General Index,2014-05-01,2.0,2014,Q2,2014-Q2


### Inflation Issues and Cleaning Decisions--Hanin

| Issue | Decision |
|---|---|
| CSV contains metadata rows before the actual header | Skip the first 4 rows when loading the file |
| Dataset contains many CPI item categories | Keep only the General Index (`Item Code = 0`) |
| Date column is stored as text | Convert `YEAR_MONTH` to datetime |
| Dataset starts in 2014 | Keep all available records from 2014 onward |
| Column names are not standardized | Convert column names to lowercase snake_case |
| Metadata columns are constant after filtering | Remove `CL_ADMIN_PRICES_E`, `D_BASKET_LVL`, and `Indicator Name` |
| Column names are not clear for downstream use | Rename `YEAR_MONTH` to `date` and `OBS_VALUE` to `inflation_rate` |
| Economic, stock, and currency datasets will need a common time reference for future joins | Add `year`, `quarter`, and `year_quarter` while keeping the original monthly `date` |
| Inflation data is monthly while GDP and unemployment are quarterly | Keep monthly records in Task 2 and convert inflation to quarterly average later in Task 4 |

In [26]:
# Load the raw unemployment CSV file
unemployment_raw = pd.read_csv(unemployment_file)

print("Unemployment shape:", unemployment_raw.shape)

unemployment_raw.head()

Unemployment shape: (477, 12)


,cl_nationality_gen_ord,cl_nationality_gen_en,cl_sex_gen_ord,cl_sex_gen_en,period,cl_quarter_gen_ord,cl_quarter_gen_en,cl_indicators_lf_en,cl_age_group_10_lf_en,obs_value,data_source,is_estimated
0,1.0,Saudi,1.0,Male,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,7.26,estimated_not_official,True
1,1.0,Saudi,2.0,Female,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,33.60,estimated_not_official,True
2,1.0,Saudi,999.0,Total,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,21.45,estimated_not_official,True
3,2.0,Non-Saudi,1.0,Male,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,0.94,estimated_not_official,True
4,2.0,Non-Saudi,2.0,Female,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,2.46,estimated_not_official,True


In [27]:
# Check the unemployment dataset structure and data types
print("Number of rows:", unemployment_raw.shape[0])
print("Number of columns:", unemployment_raw.shape[1])

unemployment_raw.info()

Number of rows: 477
Number of columns: 12
<class 'pandas.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cl_nationality_gen_ord  477 non-null    float64
 1   cl_nationality_gen_en   477 non-null    str    
 2   cl_sex_gen_ord          477 non-null    float64
 3   cl_sex_gen_en           477 non-null    str    
 4   period                  477 non-null    int64  
 5   cl_quarter_gen_ord      477 non-null    float64
 6   cl_quarter_gen_en       477 non-null    str    
 7   cl_indicators_lf_en     477 non-null    str    
 8   cl_age_group_10_lf_en   477 non-null    str    
 9   obs_value               477 non-null    float64
 10  data_source             477 non-null    str    
 11  is_estimated            477 non-null    bool   
dtypes: bool(1), float64(4), int64(1), str(6)
memory usage: 41.6 KB


In [28]:
# Check missing values in each column
unemployment_raw.isnull().sum()

cl_nationality_gen_ord    0
cl_nationality_gen_en     0
cl_sex_gen_ord            0
cl_sex_gen_en             0
period                    0
cl_quarter_gen_ord        0
cl_quarter_gen_en         0
cl_indicators_lf_en       0
cl_age_group_10_lf_en     0
obs_value                 0
data_source               0
is_estimated              0
dtype: int64

In [29]:
# Check the number of unique values in each column
unemployment_raw.nunique()

cl_nationality_gen_ord      3
cl_nationality_gen_en       3
cl_sex_gen_ord              3
cl_sex_gen_en               3
period                     14
cl_quarter_gen_ord          4
cl_quarter_gen_en           4
cl_indicators_lf_en         1
cl_age_group_10_lf_en       1
obs_value                 265
data_source                 3
is_estimated                2
dtype: int64

In [30]:
# Check duplicate rows
print("Duplicate rows:", unemployment_raw.duplicated().sum())

Duplicate rows: 0


In [31]:
# Create a profiling summary for the unemployment dataset
unemployment_profile = pd.DataFrame({
    "column_name": unemployment_raw.columns,
    "dtype": [unemployment_raw[col].dtype for col in unemployment_raw.columns],
    "null_count": [unemployment_raw[col].isnull().sum() for col in unemployment_raw.columns],
    "null_percentage": [unemployment_raw[col].isnull().mean() * 100 for col in unemployment_raw.columns],
    "unique_count": [unemployment_raw[col].nunique() for col in unemployment_raw.columns],
    "sample_value": [unemployment_raw[col].iloc[0] for col in unemployment_raw.columns]
})

unemployment_profile

,column_name,dtype,null_count,null_percentage,unique_count,sample_value
0,cl_nationality_gen_ord,float64,0,0.0,3,1.0
1,cl_nationality_gen_en,str,0,0.0,3,Saudi
2,cl_sex_gen_ord,float64,0,0.0,3,1.0
3,cl_sex_gen_en,str,0,0.0,3,Male
4,period,int64,0,0.0,14,2013
5,cl_quarter_gen_ord,float64,0,0.0,4,1.0
6,cl_quarter_gen_en,str,0,0.0,4,First Quarter
7,cl_indicators_lf_en,str,0,0.0,1,Unemployment rate by nationality and sex
8,cl_age_group_10_lf_en,str,0,0.0,1,Total
9,obs_value,float64,0,0.0,265,7.26


In [32]:
# Check the available nationality and sex categories
print("Nationality values:")
print(unemployment_raw["cl_nationality_gen_en"].unique())

print("\nSex values:")
print(unemployment_raw["cl_sex_gen_en"].unique())

Nationality values:
<StringArray>
['Saudi', 'Non-Saudi', 'Total']
Length: 3, dtype: str

Sex values:
<StringArray>
['Male', 'Female', 'Total']
Length: 3, dtype: str


In [33]:
# Keep only the overall unemployment rate
unemployment_clean = unemployment_raw[
    (unemployment_raw["cl_nationality_gen_en"] == "Total") &
    (unemployment_raw["cl_sex_gen_en"] == "Total")
].copy()

# Check the filtered data
print("Unemployment records:", unemployment_clean.shape[0])

unemployment_clean.head()

Unemployment records: 53


,cl_nationality_gen_ord,cl_nationality_gen_en,cl_sex_gen_ord,cl_sex_gen_en,period,cl_quarter_gen_ord,cl_quarter_gen_en,cl_indicators_lf_en,cl_age_group_10_lf_en,obs_value,data_source,is_estimated
8,999.0,Total,999.0,Total,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,5.88,estimated_not_official,True
17,999.0,Total,999.0,Total,2013,2.0,Second Quarter,Unemployment rate by nationality and sex,Total,5.72,estimated_not_official,True
26,999.0,Total,999.0,Total,2013,3.0,Third Quarter,Unemployment rate by nationality and sex,Total,5.69,estimated_not_official,True
35,999.0,Total,999.0,Total,2013,4.0,Fourth Quarter,Unemployment rate by nationality and sex,Total,5.66,estimated_not_official,True
44,999.0,Total,999.0,Total,2014,1.0,First Quarter,Unemployment rate by nationality and sex,Total,5.61,estimated_not_official,True


In [34]:
# Create standardized time columns for future joins
unemployment_clean["year"] = unemployment_clean["period"].astype(int)

unemployment_clean["quarter"] = (
    "Q" + unemployment_clean["cl_quarter_gen_ord"].astype(int).astype(str)
)

unemployment_clean["year_quarter"] = (
    unemployment_clean["year"].astype(str)
    + "-"
    + unemployment_clean["quarter"]
)

# Sort the data chronologically
unemployment_clean = unemployment_clean.sort_values(
    by=["year", "cl_quarter_gen_ord"]
).reset_index(drop=True)

# Display the updated data
unemployment_clean.head()

,cl_nationality_gen_ord,cl_nationality_gen_en,cl_sex_gen_ord,cl_sex_gen_en,period,cl_quarter_gen_ord,cl_quarter_gen_en,cl_indicators_lf_en,cl_age_group_10_lf_en,obs_value,data_source,is_estimated,year,quarter,year_quarter
0,999.0,Total,999.0,Total,2013,1.0,First Quarter,Unemployment rate by nationality and sex,Total,5.88,estimated_not_official,True,2013,Q1,2013-Q1
1,999.0,Total,999.0,Total,2013,2.0,Second Quarter,Unemployment rate by nationality and sex,Total,5.72,estimated_not_official,True,2013,Q2,2013-Q2
2,999.0,Total,999.0,Total,2013,3.0,Third Quarter,Unemployment rate by nationality and sex,Total,5.69,estimated_not_official,True,2013,Q3,2013-Q3
3,999.0,Total,999.0,Total,2013,4.0,Fourth Quarter,Unemployment rate by nationality and sex,Total,5.66,estimated_not_official,True,2013,Q4,2013-Q4
4,999.0,Total,999.0,Total,2014,1.0,First Quarter,Unemployment rate by nationality and sex,Total,5.61,estimated_not_official,True,2014,Q1,2014-Q1


In [35]:
# Keep the columns needed for analysis, data quality, and future joins
unemployment_clean = unemployment_clean[
    [
        "cl_indicators_lf_en",
        "obs_value",
        "is_estimated",
        "year",
        "quarter",
        "year_quarter"
    ]
].copy()

# Rename columns to clearer names
unemployment_clean = unemployment_clean.rename(
    columns={
        "cl_indicators_lf_en": "indicator_name",
        "obs_value": "unemployment_rate"
    }
)

# Display the cleaned unemployment data
unemployment_clean.head()

,indicator_name,unemployment_rate,is_estimated,year,quarter,year_quarter
0,Unemployment rate by nationality and sex,5.88,True,2013,Q1,2013-Q1
1,Unemployment rate by nationality and sex,5.72,True,2013,Q2,2013-Q2
2,Unemployment rate by nationality and sex,5.69,True,2013,Q3,2013-Q3
3,Unemployment rate by nationality and sex,5.66,True,2013,Q4,2013-Q4
4,Unemployment rate by nationality and sex,5.61,True,2014,Q1,2014-Q1


### Unemployment Issues and Cleaning Decisions-Hanin 

| Issue | Decision |
|---|---|
| Dataset contains Saudi, Non-Saudi, and Total nationality categories | Keep `Total` to represent the overall unemployment rate |
| Dataset contains Male, Female, and Total sex categories | Keep `Total` to represent both sexes |
| Dataset contains multiple age groups | Keep records where the age group is `Total` |
| Quarter is stored as an ordinal number | Standardize it to `Q1`–`Q4` |
| Time fields need to support future joins | Add standardized `year`, `quarter`, and `year_quarter` columns |
| Original year and quarter columns duplicate the standardized time fields | Remove them after creating `year`, `quarter`, and `year_quarter` |
| `OBS_VALUE` is not descriptive | Rename it to `unemployment_rate` |
| Some historical records are estimated | Keep `is_estimated` as a data-quality flag |
| `data_source` is not required in the cleaned analytical dataset | Remove it from the cleaned dataset |

In [36]:
# Display all cleaned economic indicator tables

print("===== CLEANED GDP =====")
display(gdp_clean)

print("\n===== CLEANED INFLATION =====")
display(inflation_clean)

print("\n===== CLEANED UNEMPLOYMENT =====")
display(unemployment_clean)

===== CLEANED GDP =====


,main_activities,year,quarter,year_quarter,gdp_value
0,Gross Domestic Product,2013,Q1,2013-Q1,712484.0
1,Gross Domestic Product,2013,Q2,2013-Q2,721647.0
2,Gross Domestic Product,2013,Q3,2013-Q3,729632.0
3,Gross Domestic Product,2013,Q4,2013-Q4,722821.0
4,Gross Domestic Product,2014,Q1,2014-Q1,762832.0
5,Gross Domestic Product,2014,Q2,2014-Q2,763677.0
6,Gross Domestic Product,2014,Q3,2014-Q3,759291.0
7,Gross Domestic Product,2014,Q4,2014-Q4,666024.0
8,Gross Domestic Product,2015,Q1,2015-Q1,669419.0
9,Gross Domestic Product,2015,Q2,2015-Q2,684728.0



===== CLEANED INFLATION =====


,item_name,date,inflation_rate,year,quarter,year_quarter
0,General Index,2014-01-01,1.1,2014,Q1,2014-Q1
1,General Index,2014-02-01,1.5,2014,Q1,2014-Q1
2,General Index,2014-03-01,1.5,2014,Q1,2014-Q1
3,General Index,2014-04-01,2.0,2014,Q2,2014-Q2
4,General Index,2014-05-01,2.0,2014,Q2,2014-Q2
...,...,...,...,...,...,...
146,General Index,2026-03-01,1.8,2026,Q1,2026-Q1
147,General Index,2026-04-01,1.7,2026,Q2,2026-Q2
148,General Index,2026-05-01,1.8,2026,Q2,2026-Q2
149,General Index,2026-06-01,1.8,2026,Q2,2026-Q2



===== CLEANED UNEMPLOYMENT =====


,indicator_name,unemployment_rate,is_estimated,year,quarter,year_quarter
0,Unemployment rate by nationality and sex,5.88,True,2013,Q1,2013-Q1
1,Unemployment rate by nationality and sex,5.72,True,2013,Q2,2013-Q2
2,Unemployment rate by nationality and sex,5.69,True,2013,Q3,2013-Q3
3,Unemployment rate by nationality and sex,5.66,True,2013,Q4,2013-Q4
4,Unemployment rate by nationality and sex,5.61,True,2014,Q1,2014-Q1
5,Unemployment rate by nationality and sex,5.70,True,2014,Q2,2014-Q2
6,Unemployment rate by nationality and sex,5.64,True,2014,Q3,2014-Q3
7,Unemployment rate by nationality and sex,5.72,True,2014,Q4,2014-Q4
8,Unemployment rate by nationality and sex,5.68,True,2015,Q1,2015-Q1
9,Unemployment rate by nationality and sex,5.74,True,2015,Q2,2015-Q2


In [37]:
# Check the final cleaned datasets before saving

print("GDP:")
print("Shape:", gdp_clean.shape)
print("Columns:", gdp_clean.columns.tolist())

print("\nInflation:")
print("Shape:", inflation_clean.shape)
print("Columns:", inflation_clean.columns.tolist())

print("\nUnemployment:")
print("Shape:", unemployment_clean.shape)
print("Columns:", unemployment_clean.columns.tolist())

GDP:
Shape: (53, 5)
Columns: ['main_activities', 'year', 'quarter', 'year_quarter', 'gdp_value']

Inflation:
Shape: (151, 6)
Columns: ['item_name', 'date', 'inflation_rate', 'year', 'quarter', 'year_quarter']

Unemployment:
Shape: (53, 6)
Columns: ['indicator_name', 'unemployment_rate', 'is_estimated', 'year', 'quarter', 'year_quarter']


In [38]:
# Save each cleaned dataset separately for the next tasks

gdp_clean.to_csv(
    INTERIM_DATA_PATH / "gdp_cleaned.csv",
    index=False
)

inflation_clean.to_csv(
    INTERIM_DATA_PATH / "inflation_cleaned.csv",
    index=False,
    date_format="%d-%m-%Y"
)

unemployment_clean.to_csv(
    INTERIM_DATA_PATH / "unemployment_cleaned.csv",
    index=False
)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [39]:
# Check for mixed data types in each dataset

def check_mixed_types(df, dataset_name):
    print(f"\n{dataset_name}")
    
    for column in df.columns:
        types = df[column].dropna().map(type).unique()

        if len(types) > 1:
            print(f"{column}: Mixed types found")
        else:
            print(f"{column}: No mixed types")


check_mixed_types(gdp_raw, "GDP")
check_mixed_types(inflation_raw, "Inflation")
check_mixed_types(unemployment_raw, "Unemployment")


GDP
Item Code: No mixed types
Main Activities: No mixed types
Year / Quarter: No mixed types
Gross Domestic Product: No mixed types

Inflation
Item Code: No mixed types
Item Name: No mixed types
YEAR_MONTH: No mixed types
CL_ADMIN_PRICES_E: No mixed types
D_BASKET_LVL: No mixed types
Indicator Name: No mixed types
OBS_VALUE: No mixed types

Unemployment
cl_nationality_gen_ord: No mixed types
cl_nationality_gen_en: No mixed types
cl_sex_gen_ord: No mixed types
cl_sex_gen_en: No mixed types
period: No mixed types
cl_quarter_gen_ord: No mixed types
cl_quarter_gen_en: No mixed types
cl_indicators_lf_en: No mixed types
cl_age_group_10_lf_en: No mixed types
obs_value: No mixed types
data_source: No mixed types
is_estimated: No mixed types


### Mixed Data Types Check- Hanin

No mixed data types were found in the GDP, Inflation, or Unemployment datasets.

In [40]:
gdp_clean.info()
gdp_clean.isnull().sum()
gdp_clean.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_activities  53 non-null     str    
 1   year             53 non-null     int64  
 2   quarter          53 non-null     str    
 3   year_quarter     53 non-null     str    
 4   gdp_value        53 non-null     float64
dtypes: float64(1), int64(1), str(3)
memory usage: 2.2 KB


np.int64(0)